# Feature Fusion Extraction — Motion (LK) + Visual (RF-DETR, DINOv2, VideoMAE, OpenCLIP)

Builds one fused feature vector per clip. **Branch-selectable** (see
`BRANCHES` in Step 0): run only the cheap branches (LK + RF-DETR --
no deep embedding models, fast even on CPU) or only the expensive ones
(DINOv2 + VideoMAE + OpenCLIP -- GPU strongly recommended) or any subset.

**LK branch rewritten 2026-10-05** -- two fixes, both verified as real
issues before being written (not assumed): (1) frames are now resized to
a fixed size before flow computation, since raw pixel-magnitude flow
wasn't comparable across clips of different native resolutions (XD-
Violence mixes movie and YouTube sources at very different resolutions);
(2) motion is now sampled from 3 windows spread across the clip (start/
~40%/~80%) instead of only the first ~2-3 seconds, since a single early
window could easily miss where the actual escalation happens. This
targets the branch ablation finding (`docs/findings.md`, 2026-09-27
entries) that LK alone sat at the trivial majority-class baseline --
**not yet re-run/validated**, this is the fix, not a result.

**OpenCLIP branch added 2026-10-05** -- a second, larger/different frozen
vision backbone (ViT-L-14, `open_clip`, same library already used for
this project's zero-shot CLIP work) alongside DINOv2, per the pure-
brainstorm question of whether another backbone could match or surpass
it. Same pattern as the DINOv2 branch: encode sparse frames, mean-pool.

Output filename reflects which branches were run this session
(`features_<branches>.csv`) -- so different partial runs don't collide,
and `ablation_fusion_branches.ipynb` can find whichever combination(s)
you've actually run.

**Requires:** both pull notebooks already run (again, if you deleted
the Drive data after the last run) --
`My Drive/CVPHG/xdviolence_riot/` and `My Drive/CVPHG/xdviolence_normal/`
each populated with their manifest CSV.

**Resumable:** re-running with the same `BRANCHES` selection skips any
`video_id` already present in that selection's output CSV.

## Step 0 — Setup and branch selection

**Edit `BRANCHES` below, then Runtime -> Change runtime type -> T4 GPU**
if it includes `'dinov2'`, `'videomae'`, or `'openclip'` (recommended --
CPU is what made the full DINOv2/VideoMAE run take 3.45 hours last time).
`{'lk', 'rfdetr'}` alone runs fine on CPU.

In [ ]:
!pip install rfdetr supervision opencv-python-headless transformers open_clip_torch -q

In [ ]:
# Edit this set to choose what this run computes. Valid values:
# 'lk', 'rfdetr', 'dinov2', 'videomae', 'openclip'. Any subset is fine.
BRANCHES = {'lk', 'rfdetr'}

import csv
import os

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

RIOT_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_riot'
NORMAL_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_normal'
OUT_DIR = '/content/drive/MyDrive/CVPHG/fusion_features'
os.makedirs(OUT_DIR, exist_ok=True)

BRANCH_ORDER = ['lk', 'rfdetr', 'dinov2', 'videomae', 'openclip']
branch_suffix = '_'.join(b for b in BRANCH_ORDER if b in BRANCHES)
OUT_CSV = os.path.join(OUT_DIR, f'features_{branch_suffix}.csv')
print('This run computes:', sorted(BRANCHES))
print('Output file:', OUT_CSV)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if ({'dinov2', 'videomae', 'openclip'} & BRANCHES) and DEVICE == 'cpu':
    print('WARNING: a GPU-preferred branch is selected but no GPU detected -- this will be slow. '
          'Runtime -> Change runtime type -> T4 GPU, then re-run this cell.')

In [ ]:
riot_manifest = pd.read_csv(os.path.join(RIOT_DIR, 'manifest_xdviolence_riot.csv'))
riot_manifest['clip_dir'] = RIOT_DIR
normal_manifest = pd.read_csv(os.path.join(NORMAL_DIR, 'manifest_xdviolence_normal.csv'))
normal_manifest['clip_dir'] = NORMAL_DIR

all_clips = pd.concat([riot_manifest, normal_manifest], ignore_index=True)
print(f"{len(riot_manifest)} riot + {len(normal_manifest)} normal = {len(all_clips)} clips total")

## Step 1 — Load only the models this run needs

In [ ]:
if 'rfdetr' in BRANCHES:
    from rfdetr import RFDETRMedium
    rfdetr_model = RFDETRMedium()
    PERSON_CLASS_ID = 0
    VEHICLE_CLASS_IDS = {1, 2, 3, 5, 7}  # bicycle, car, motorcycle, bus, truck
    DETECTION_THRESHOLD = 0.5

In [ ]:
if 'dinov2' in BRANCHES:
    dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    dinov2_model.eval().to(DEVICE)

    DINOV2_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    DINOV2_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def dinov2_embed_frame(frame_rgb: np.ndarray) -> np.ndarray:
        """frame_rgb: HxWx3 uint8. Returns a 768-d CLS embedding."""
        img = cv2.resize(frame_rgb, (224, 224), interpolation=cv2.INTER_AREA)
        tensor = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) / 255.0
        tensor = (tensor - DINOV2_MEAN) / DINOV2_STD
        tensor = tensor.to(DEVICE)
        with torch.no_grad():
            feat = dinov2_model(tensor)  # (1, 768) pooled CLS embedding
        return feat.squeeze(0).cpu().numpy()

In [ ]:
if 'videomae' in BRANCHES:
    from transformers import VideoMAEModel, VideoMAEVideoProcessor
    videomae_processor = VideoMAEVideoProcessor.from_pretrained('MCG-NJU/videomae-base')
    videomae_model = VideoMAEModel.from_pretrained('MCG-NJU/videomae-base').eval().to(DEVICE)
    VIDEOMAE_NUM_FRAMES = 16  # required by this checkpoint's config (num_frames=16)

    def videomae_embed_clip(frames_16_rgb: list) -> np.ndarray:
        """frames_16_rgb: list of exactly 16 HxWx3 uint8 RGB frames.
        Returns a 768-d mean-pooled clip embedding."""
        inputs = videomae_processor(frames_16_rgb, return_tensors='pt')
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = videomae_model(**inputs)
        # last_hidden_state: (1, 1568, 768) -- mean-pool over all tokens for one clip vector
        return outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

In [ ]:
if 'openclip' in BRANCHES:
    import open_clip
    openclip_model, _, openclip_preprocess = open_clip.create_model_and_transforms(
        'ViT-L-14', pretrained='openai'
    )
    openclip_model.eval().to(DEVICE)
    OPENCLIP_DIM = 768

    def openclip_embed_frame(frame_rgb: np.ndarray) -> np.ndarray:
        """frame_rgb: HxWx3 uint8. Returns a 768-d normalized embedding."""
        img = Image.fromarray(frame_rgb)
        tensor = openclip_preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feat = openclip_model.encode_image(tensor)
            feat = feat / feat.norm(dim=-1, keepdim=True)
        return feat.squeeze(0).cpu().numpy()

## Step 2 — Motion branch: Lucas-Kanade optical flow (self-contained, rewritten 2026-10-05)

Mirrors `src/optical_flow.py`'s core LK logic but **no longer matches it
exactly** -- two deliberate changes versus the original (both the local
pipeline and the first version of this notebook):

1. **Frames are resized to `LK_RESIZE`x`LK_RESIZE` before flow
   computation.** Verified directly (not assumed) that the original
   version never resized -- raw pixel-magnitude flow was being compared
   across clips of very different native resolutions (movie vs. YouTube
   sources), which isn't a meaningful comparison.
2. **Motion is sampled from 3 windows spread across the clip** (start,
   ~40%, ~80% of total duration) instead of only the first ~2-3 seconds
   from the very start -- a single early window could miss wherever the
   actual escalation happens in a longer clip. Per-window stats are
   pooled before the final mean/max/angle/tracked aggregation, so the
   output is still 4 scalars (same schema as before), just informed by a
   much larger share of each clip.

This targets the 2026-09-27 ablation finding that LK alone sat at the
trivial majority-class baseline -- this is the fix, not yet a validated
result.

In [ ]:
LK_FEATURE_PARAMS = dict(maxCorners=200, qualityLevel=0.01, minDistance=7, blockSize=7)
LK_WIN_SIZE = (15, 15)
LK_MAX_LEVEL = 2
LK_CRITERIA = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
LK_RESIZE = 256  # fixed size before flow computation -- makes pixel-magnitude flow comparable across clips of different native resolutions (fix 1)
LK_WINDOW_FRAMES = 32  # consecutive frames read per window
LK_STRIDE = 2
LK_WINDOW_STARTS_FRAC = (0.0, 0.4, 0.8)  # spread windows across the clip, not just the start (fix 2)


def get_frame_count(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return total


def sample_consecutive_gray(video_path, start_frame=0, max_frames=LK_WINDOW_FRAMES, stride=LK_STRIDE):
    cap = cv2.VideoCapture(video_path)
    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames = []
    idx = 0
    while len(frames) < max_frames:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        if idx % stride == 0:
            resized = cv2.resize(frame_bgr, (LK_RESIZE, LK_RESIZE), interpolation=cv2.INTER_AREA)
            frames.append(cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY))
        idx += 1
    cap.release()
    return frames


def flow_stats_for_window(frames_gray):
    """Returns raw per-frame-pair lists (not yet aggregated to scalars),
    so multiple windows can be pooled before final aggregation."""
    mags, max_mags, angles, tracked_counts = [], [], [], []
    if len(frames_gray) < 2:
        return mags, max_mags, angles, tracked_counts

    prev_gray = frames_gray[0]
    prev_pts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **LK_FEATURE_PARAMS)

    for i in range(1, len(frames_gray)):
        curr_gray = frames_gray[i]
        if prev_pts is None or len(prev_pts) == 0:
            prev_pts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **LK_FEATURE_PARAMS)
        if prev_pts is None or len(prev_pts) == 0:
            prev_gray = curr_gray
            continue

        next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
            prev_gray, curr_gray, prev_pts, None,
            winSize=LK_WIN_SIZE, maxLevel=LK_MAX_LEVEL, criteria=LK_CRITERIA,
        )
        status = status.reshape(-1)
        good_prev = prev_pts.reshape(-1, 2)[status == 1]
        good_next = next_pts.reshape(-1, 2)[status == 1]

        if len(good_next) > 0:
            flow_vecs = good_next - good_prev
            m = np.linalg.norm(flow_vecs, axis=1)
            a = np.degrees(np.arctan2(flow_vecs[:, 1], flow_vecs[:, 0])) % 360
            mags.append(float(m.mean()))
            max_mags.append(float(m.max()))
            angles.append(float(a.mean()))
            tracked_counts.append(len(good_next))

        prev_gray = curr_gray
        prev_pts = good_next.reshape(-1, 1, 2).astype(np.float32) if len(good_next) > 0 else None

    return mags, max_mags, angles, tracked_counts


def compute_lk_summary(video_path):
    total_frames = get_frame_count(video_path)
    if total_frames < 2:
        return dict(lk_mean_magnitude=0.0, lk_max_magnitude=0.0, lk_mean_angle_deg=0.0, lk_mean_tracked=0.0)

    pooled_mags, pooled_max_mags, pooled_angles, pooled_tracked = [], [], [], []
    for frac in LK_WINDOW_STARTS_FRAC:
        start_frame = int(frac * total_frames)
        if start_frame >= total_frames - 1:
            continue
        frames_gray = sample_consecutive_gray(video_path, start_frame=start_frame)
        m, mm, a, t = flow_stats_for_window(frames_gray)
        pooled_mags.extend(m)
        pooled_max_mags.extend(mm)
        pooled_angles.extend(a)
        pooled_tracked.extend(t)

    if not pooled_mags:
        return dict(lk_mean_magnitude=0.0, lk_max_magnitude=0.0, lk_mean_angle_deg=0.0, lk_mean_tracked=0.0)
    return dict(
        lk_mean_magnitude=float(np.mean(pooled_mags)), lk_max_magnitude=float(np.max(pooled_max_mags)),
        lk_mean_angle_deg=float(np.mean(pooled_angles)), lk_mean_tracked=float(np.mean(pooled_tracked)),
    )

## Step 3 — Visual branch helpers

In [ ]:
N_SPARSE_FRAMES = 4  # for RF-DETR + DINOv2 (evenly spaced across the clip)


def sample_frames_rgb(video_path, n):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    idxs = np.linspace(0, total - 1, min(n, total)).astype(int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame_bgr = cap.read()
        if ok:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def compute_rfdetr_summary(frames_rgb):
    if not frames_rgb:
        return dict(rfdetr_mean_person=0.0, rfdetr_max_person=0.0,
                    rfdetr_mean_vehicle=0.0, rfdetr_mean_confidence=0.0)
    persons, vehicles, confs = [], [], []
    for frame in frames_rgb:
        det = rfdetr_model.predict(Image.fromarray(frame), threshold=DETECTION_THRESHOLD)
        persons.append(int((det.class_id == PERSON_CLASS_ID).sum()))
        vehicles.append(int(np.isin(det.class_id, list(VEHICLE_CLASS_IDS)).sum()))
        confs.append(float(det.confidence.mean()) if len(det) else 0.0)
    return dict(
        rfdetr_mean_person=float(np.mean(persons)), rfdetr_max_person=float(np.max(persons)),
        rfdetr_mean_vehicle=float(np.mean(vehicles)), rfdetr_mean_confidence=float(np.mean(confs)),
    )

## Step 4 — Per-clip extraction loop (resumable, branch-selective)

In [ ]:
DINOV2_DIM = 768
VIDEOMAE_DIM = 768
OPENCLIP_DIM = 768

FIELDNAMES = ['video_id', 'label', 'split']
if 'lk' in BRANCHES:
    FIELDNAMES += ['lk_mean_magnitude', 'lk_max_magnitude', 'lk_mean_angle_deg', 'lk_mean_tracked']
if 'rfdetr' in BRANCHES:
    FIELDNAMES += ['rfdetr_mean_person', 'rfdetr_max_person', 'rfdetr_mean_vehicle', 'rfdetr_mean_confidence']
if 'dinov2' in BRANCHES:
    FIELDNAMES += [f'dinov2_{i}' for i in range(DINOV2_DIM)]
if 'videomae' in BRANCHES:
    FIELDNAMES += [f'videomae_{i}' for i in range(VIDEOMAE_DIM)]
if 'openclip' in BRANCHES:
    FIELDNAMES += [f'openclip_{i}' for i in range(OPENCLIP_DIM)]

already_done = set()
if os.path.exists(OUT_CSV):
    already_done = set(pd.read_csv(OUT_CSV, usecols=['video_id'])['video_id'])
    print(f"Resuming: {len(already_done)} clips already extracted for this branch selection.")

write_header = not os.path.exists(OUT_CSV)
with open(OUT_CSV, 'a', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=FIELDNAMES)
    if write_header:
        writer.writeheader()

    for i, row in enumerate(all_clips.to_dict('records'), 1):
        if row['id'] in already_done:
            continue
        video_path = os.path.join(row['clip_dir'], row['id'])
        try:
            record = {'video_id': row['id'], 'label': row['label'], 'split': row['split']}

            if 'lk' in BRANCHES:
                record.update(compute_lk_summary(video_path))

            need_sparse = 'rfdetr' in BRANCHES or 'dinov2' in BRANCHES or 'openclip' in BRANCHES
            sparse_frames = sample_frames_rgb(video_path, N_SPARSE_FRAMES) if need_sparse else []

            if 'rfdetr' in BRANCHES:
                record.update(compute_rfdetr_summary(sparse_frames))

            if 'dinov2' in BRANCHES:
                dinov2_vec = (np.mean([dinov2_embed_frame(f) for f in sparse_frames], axis=0)
                              if sparse_frames else np.zeros(DINOV2_DIM))
                record.update({f'dinov2_{j}': float(v) for j, v in enumerate(dinov2_vec)})

            if 'openclip' in BRANCHES:
                openclip_vec = (np.mean([openclip_embed_frame(f) for f in sparse_frames], axis=0)
                                if sparse_frames else np.zeros(OPENCLIP_DIM))
                record.update({f'openclip_{j}': float(v) for j, v in enumerate(openclip_vec)})

            if 'videomae' in BRANCHES:
                videomae_frames = sample_frames_rgb(video_path, VIDEOMAE_NUM_FRAMES)
                if len(videomae_frames) < VIDEOMAE_NUM_FRAMES:
                    videomae_frames = (videomae_frames * VIDEOMAE_NUM_FRAMES)[:VIDEOMAE_NUM_FRAMES] if videomae_frames else []
                videomae_vec = videomae_embed_clip(videomae_frames) if videomae_frames else np.zeros(VIDEOMAE_DIM)
                record.update({f'videomae_{j}': float(v) for j, v in enumerate(videomae_vec)})

            writer.writerow(record)
            fh.flush()
        except Exception as e:
            print(f"  FAILED {row['id']}: {e}")
            continue

        if i % 25 == 0 or i == len(all_clips):
            print(f"  {i}/{len(all_clips)} clips processed")

print(f"\nDone. Features saved to {OUT_CSV}")

## Next steps

- Run this notebook once per branch group you need: `{'lk', 'rfdetr'}`
  (cheap, CPU is fine -- note the LK branch is the 2026-10-05 rewrite,
  not yet validated), `{'dinov2', 'videomae', 'openclip'}` (GPU
  recommended), and optionally all together to reproduce/extend the
  full-fusion number.
- Feed whichever `features_*.csv` files you produce into
  `ablation_fusion_branches.ipynb` to compare them -- it already knows
  about an `openclip_only` group (added 2026-10-05).
- First thing to check once the LK rewrite is run: does `lk_only` move
  off the 82.89% trivial-baseline floor at all? That's the direct test
  of whether the resize + multi-window fix actually helped.
- If VideoMAE padding (repeating frames for short clips) shows up a lot
  in practice, worth checking clip-length distribution -- not checked
  yet in this notebook.